In [1]:
# Code for 401(k) example illustrating Stacking (PyTorch)
# Ten candidate learners aggregated via 5-fold CV with stacking weights
# (unconstrained least squares + non-negative-sum-to-one constrained QP).
# Candidate-learner architectures aligned to the Stata pystacked specification
# (which is the canonical source for the slide's stacking results table).

import os, copy, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from scipy.optimize import minimize

import warnings
warnings.filterwarnings("ignore")

SEED = 8261977
np.random.seed(SEED)

# Load 401(k) data
data = pd.read_csv(os.path.join("..", "Data", "restatw.dat"), sep=r"\s+")
data = data[["net_tfa", "e401", "age", "inc", "educ", "fsize",
             "marr", "twoearn", "db", "pira", "hown"]]

# Normalize select columns
data["age"]   /= 64
data["inc"]   /= 250000
data["fsize"] /= 13
data["educ"]  /= 18

# Train/test split (8000 train, rest test)
ntrain = 8000
tr = np.random.choice(len(data), ntrain, replace=False)
te = np.setdiff1d(np.arange(len(data)), tr)
train = data.iloc[tr].reset_index(drop=True)
test  = data.iloc[te].reset_index(drop=True)

ytr = train["net_tfa"].values
yte = test["net_tfa"].values
print(f"Train n = {len(train)}, Test n = {len(test)}")


Train n = 8000, Test n = 1915


In [2]:
# Build HDLM design matrix:
#   baseline features + poly(age,6) + poly(inc,8) + poly(educ,4) + poly(fsize,2),
#   with all pairwise interactions (mirrors R's
#   net_tfa ~ (e401 + poly(age,6) + ...)^2  via model.matrix).
base_cols = ["e401", "marr", "twoearn", "db", "pira", "hown"]

def design_matrix(df):
    cols = [df[c].values for c in base_cols]
    for col, deg in [("age", 6), ("inc", 8), ("educ", 4), ("fsize", 2)]:
        for d in range(1, deg + 1):
            cols.append(df[col].values ** d)
    X = np.column_stack(cols).astype(np.float64)
    n, p = X.shape
    int_cols = [X[:, i] * X[:, j] for i in range(p) for j in range(i + 1, p)]
    return np.column_stack([X] + int_cols)

xtr = design_matrix(train)
xte = design_matrix(test)
print(f"HDLM design matrix: train {xtr.shape}, test {xte.shape}")

# Raw 10-feature matrices (for OLS-base, RF, xgboost, DNN)
feature_cols = ["e401", "age", "inc", "educ", "fsize",
                "marr", "twoearn", "db", "pira", "hown"]
Xall_tr = train[feature_cols].values
Xall_te = test[feature_cols].values


HDLM design matrix: train (8000, 351), test (1915, 351)


In [3]:
# PyTorch DNN helper (plain MLP with ReLU; matches sklearn MLPRegressor's
# hidden_layer_sizes specification used in the Stata pystacked code).
def _t(a):
    a = np.asarray(a, dtype=np.float32)
    if a.ndim == 1:
        a = a.reshape(-1, 1)
    return torch.from_numpy(a)

def make_mlp(p, hidden_layer_sizes):
    layers, prev = [], p
    for h in hidden_layer_sizes:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        prev = h
    layers.append(nn.Linear(prev, 1))
    return nn.Sequential(*layers)

def fit_dnn_predict(X_train, y_train, X_pred_list, hidden,
                    epochs=200, batch_size=200, lr=1e-3, seed=720):
    torch.manual_seed(int(seed))
    model = make_mlp(X_train.shape[1], hidden)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    sx, sy = _t(X_train), _t(y_train)
    dl = DataLoader(TensorDataset(sx, sy), batch_size=batch_size, shuffle=True,
                    generator=torch.Generator().manual_seed(int(seed)))
    for _ in range(epochs):
        model.train()
        for xb, yb in dl:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return [model(_t(X)).numpy().flatten() for X in X_pred_list]


In [4]:
# 5-fold CV loop building yhat_tr (in-fold) and yhat_te (out-of-fold averaged)
ntr, nte, Kf = len(ytr), len(yte), 5
np.random.seed(SEED)
cvgroup = np.random.permutation(np.tile(np.arange(1, Kf + 1),
                                        int(np.ceil(ntr / Kf)))[:ntr])

yhat_tr = np.zeros((ntr, 10))
yhat_te = np.zeros((nte, 10))

dnn_specs = [(50, 10, 50), (50, 50, 50, 50), (100, 100, 100, 100, 100)]

t0 = time.time()
for k in range(1, Kf + 1):
    indk = (cvgroup == k)
    print(f"Fold {k}/{Kf} ...", flush=True)

    # 1. OLS - basic (baseline features only)
    ols = LinearRegression().fit(Xall_tr[~indk], ytr[~indk])
    yhat_tr[indk, 0]  = ols.predict(Xall_tr[indk])
    yhat_te[:, 0]    += ols.predict(Xall_te) / Kf

    # 2. OLS - flexible (HDLM design)
    ols = LinearRegression().fit(xtr[~indk], ytr[~indk])
    yhat_tr[indk, 1]  = ols.predict(xtr[indk])
    yhat_te[:, 1]    += ols.predict(xte) / Kf

    # 3. Lasso (CV-tuned)
    lasso = LassoCV(cv=5, random_state=SEED, max_iter=20000).fit(xtr[~indk], ytr[~indk])
    yhat_tr[indk, 2]  = lasso.predict(xtr[indk])
    yhat_te[:, 2]    += lasso.predict(xte) / Kf

    # 4. Ridge (CV-tuned)
    ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(xtr[~indk], ytr[~indk])
    yhat_tr[indk, 3]  = ridge.predict(xtr[indk])
    yhat_te[:, 3]    += ridge.predict(xte) / Kf

    # 5. Random Forest
    rf = RandomForestRegressor(n_estimators=500, random_state=720,
                               min_samples_leaf=20, n_jobs=1)
    rf.fit(Xall_tr[~indk], ytr[~indk])
    yhat_tr[indk, 4]  = rf.predict(Xall_tr[indk])
    yhat_te[:, 4]    += rf.predict(Xall_te) / Kf

    # 6 & 7. Gradient-boosted trees, depth 3 and 5
    dtr_in = xgb.DMatrix(Xall_tr[~indk], label=ytr[~indk])
    dtr_oof = xgb.DMatrix(Xall_tr[indk])
    dte_full = xgb.DMatrix(Xall_te)
    for j, depth in enumerate([3, 5]):
        params = {"eta": 0.1, "max_depth": depth,
                  "objective": "reg:squarederror", "verbosity": 0}
        cv_res = xgb.cv(params, dtr_in, num_boost_round=500, nfold=5,
                        seed=720, verbose_eval=False)
        best_iter = int(cv_res["test-rmse-mean"].idxmin()) + 1
        bst = xgb.train(params, dtr_in, num_boost_round=best_iter)
        yhat_tr[indk, 5 + j]  = bst.predict(dtr_oof)
        yhat_te[:, 5 + j]    += bst.predict(dte_full) / Kf

    # 8, 9, 10. DNNs at three Stata-aligned architectures
    for j, hidden in enumerate(dnn_specs):
        preds = fit_dnn_predict(Xall_tr[~indk], ytr[~indk],
                                [Xall_tr[indk], Xall_te],
                                hidden=list(hidden), seed=720)
        yhat_tr[indk, 7 + j]  = preds[0]
        yhat_te[:, 7 + j]    += preds[1] / Kf

print(f"CV loop done in {time.time() - t0:.1f}s")


Fold 1/5 ...


Fold 2/5 ...


Fold 3/5 ...


Fold 4/5 ...


Fold 5/5 ...


CV loop done in 337.8s


In [5]:
# Stacking weights
# Unconstrained: ordinary least squares regression of y on yhat (no intercept)
from sklearn.linear_model import LinearRegression
w_unc = LinearRegression(fit_intercept=False).fit(yhat_tr, ytr).coef_

# Constrained: weights >= 0, sum to 1.
# We use NNLS (non-negative least squares) and renormalize.  NNLS is numerically
# robust to columns with extreme scale (e.g. the OLS-flexible column whose fold-out
# predictions blow up because the 351-dim HDLM design is rank-deficient on
# training subsets); it cleanly gives a zero weight to such columns.  After NNLS we
# renormalize so the weights sum to 1, matching the conventional stacking
# constraint set.
from scipy.optimize import nnls
w_nn, _ = nnls(yhat_tr, ytr, maxiter=2000)
total = w_nn.sum()
w_con = w_nn / total if total > 1e-10 else np.ones(10) / 10
print(f"NNLS+renormalize weights: sum = {w_con.sum():.4f}, "
      f"min = {w_con.min():.4f}, max = {w_con.max():.4f}")
print("Per-learner weights:")
for j, name in enumerate(["OLS-base", "OLS-flex", "Lasso", "Ridge", "RF",
                          "Boost-d3", "Boost-d5", "DNN-50/10/50",
                          "DNN-50/50/50/50", "DNN-100x5"]):
    print(f"  {name:18s}  {w_con[j]:.4f}")


NNLS+renormalize weights: sum = 1.0000, min = 0.0000, max = 0.4853
Per-learner weights:
  OLS-base            0.0000
  OLS-flex            0.0000
  Lasso               0.0000
  Ridge               0.4853
  RF                  0.0000
  Boost-d3            0.2316
  Boost-d5            0.0636
  DNN-50/10/50        0.0000
  DNN-50/50/50/50     0.0000
  DNN-100x5           0.2194


In [6]:
# Performance summary table
ymean_tr = ytr.mean()
def r2(y, yhat, baseline_var):
    return 1 - np.mean((y - yhat) ** 2) / baseline_var

sst_tr = np.mean((ytr - ymean_tr) ** 2)
sst_te = np.mean((yte - ymean_tr) ** 2)

# CV (in-fold) R^2 for each learner + the two stacked ensembles
cv_r2  = [r2(ytr, yhat_tr[:, j], sst_tr) for j in range(10)]
cv_r2 += [r2(ytr, yhat_tr @ w_unc, sst_tr),
          r2(ytr, yhat_tr @ w_con, sst_tr)]

# Test R^2 for each learner + the two stacked ensembles
te_r2  = [r2(yte, yhat_te[:, j], sst_te) for j in range(10)]
te_r2 += [r2(yte, yhat_te @ w_unc, sst_te),
          r2(yte, yhat_te @ w_con, sst_te)]

learners = [
    "OLS - basic", "OLS - flexible", "Lasso (CV)", "Ridge (CV)",
    "Random forest", "Boosted trees - depth 3", "Boosted trees - depth 5",
    "DNN - 50/10/50", "DNN - 50/50/50/50", "DNN - 100/100/100/100/100",
    "Stacking (unconstrained)", "Stacking (constrained)",
]
weights = list(w_con) + [np.nan, np.nan]

results = pd.DataFrame({
    "CV R^2":   np.round(cv_r2, 3),
    "Test R^2": np.round(te_r2, 3),
    "Weight":   np.round(weights, 3),
}, index=learners)
print(results)


                                 CV R^2  Test R^2  Weight
OLS - basic                2.330000e-01     0.193   0.000
OLS - flexible            -8.117554e+07   -11.507   0.000
Lasso (CV)                 3.340000e-01     0.199   0.000
Ridge (CV)                 3.380000e-01     0.230   0.485
Random forest              2.720000e-01     0.207   0.000
Boosted trees - depth 3    3.160000e-01     0.198   0.232
Boosted trees - depth 5    2.780000e-01     0.169   0.064
DNN - 50/10/50             2.530000e-01     0.203   0.000
DNN - 50/50/50/50          3.190000e-01     0.201   0.000
DNN - 100/100/100/100/100  3.300000e-01     0.209   0.219
Stacking (unconstrained)   3.490000e-01     0.217     NaN
Stacking (constrained)     3.450000e-01     0.222     NaN
